# RSNA Knee Abnormality Detection — Frozen Image Baseline

Builds the Phase 3B image baseline exactly as specified: one validated series per anatomical plane, five central-band slices each, physical-aspect letterboxing, DICOM-faithful intensity normalization, signed laterality canonicalization, a frozen DINOv2-small encoder, shared-mean aggregation to one 388-dimensional study vector, and a single strongly regularized multilabel linear head evaluated out-of-fold on the 58 human-labeled studies.

Every configuration value below is frozen before this notebook runs and is displayed rather than described, so the exact contract is visible in the output. Every result is an aggregate count, rate, or score — no report text, no study or series identifiers, no per-study predictions, and no pixel data is displayed or persisted.

In [ ]:
import hashlib
import importlib.metadata
import importlib.util
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError("This notebook runs on Kaggle only.")

# Verify and install the pinned offline wheel BEFORE inserting the source
# path or importing knee_mri anywhere, so the import cannot silently pick up
# a different stratifier than the one this contract pins.
package_initializers = tuple(Path("/kaggle/input/datasets").rglob("knee_mri/__init__.py"))
if len(package_initializers) != 1:
    raise RuntimeError("Expected exactly one attached knee_mri source package.")
_src_root = package_initializers[0].parent.parent
_dataset_root = _src_root.parent

WHEEL_NAME = "iterative_stratification-0.1.9-py3-none-any.whl"
EXPECTED_SHA256 = "476f8deff6753fb1725612fe41e59cc2058f8f2524ae5d1ccee88eb8c8d3de80"

wheel_matches = tuple(_dataset_root.rglob(WHEEL_NAME))
if len(wheel_matches) != 1:
    raise RuntimeError("Expected exactly one pinned iterative-stratification wheel.")
wheel_path = wheel_matches[0]
if hashlib.sha256(wheel_path.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("Pinned iterative-stratification wheel checksum mismatch.")

try:
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", str(wheel_path)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
except OSError:
    raise RuntimeError("Failed to launch the offline install for the pinned wheel.") from None
if install_result.returncode != 0:
    raise RuntimeError("Offline installation of the pinned wheel failed.")
if importlib.metadata.version("iterative-stratification") != "0.1.9":
    raise RuntimeError("Installed iterative-stratification version mismatch.")

# The processor statistics come from the vendored copy of the attached
# model's own preprocessor_config.json. There is deliberately no fallback:
# substituting remembered constants is the silent-wrongness this contract
# exists to prevent.
PROCESSOR_CONFIG_NAME = "dinov2-small-preprocessor_config.json"
processor_matches = tuple(_dataset_root.rglob(PROCESSOR_CONFIG_NAME))
if len(processor_matches) != 1:
    raise RuntimeError("Expected exactly one vendored DINOv2 processor config.")
PROCESSOR_CONFIG_PATH = processor_matches[0]

# Install the vendored DICOM codec plugins. The corpus-wide census found no
# compressed series in either released split, so these are insurance for the
# hidden set rather than a current requirement -- but an undecodable slice
# there would fail silently into the fallback row, which is the failure mode
# worth spending a few seconds to avoid.
#
# --no-deps is required, not stylistic: both compiled wheels declare
# numpy>=2.0,<3.0, and without it pip would try to resolve or replace the
# kernel's own numpy, offline and unasked.
CODEC_WHEELS = {
    "pylibjpeg-2.1.0-py3-none-any.whl":
        "25df9496a69e64e98c887fddee12a1271e275b5f74ba804f9bf98a08bb80993e",
    "pylibjpeg_openjpeg-2.5.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl":
        "a22fcb649ba9849209d8e43dba88632445a5941f0cd6765338b3652a4c686140",
    "pylibjpeg_libjpeg-2.4.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl":
        "01d950ef496476a9223e4966376cb88098fcf5c55a12a21f722d7b5f84daae43",
}

codec_paths = []
for codec_name, codec_sha256 in CODEC_WHEELS.items():
    matches = tuple(_dataset_root.rglob(codec_name))
    if len(matches) != 1:
        raise RuntimeError("Expected exactly one copy of each vendored codec wheel.")
    if hashlib.sha256(matches[0].read_bytes()).hexdigest() != codec_sha256:
        raise RuntimeError("Vendored codec wheel checksum mismatch.")
    codec_paths.append(str(matches[0]))

try:
    codec_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", *codec_paths],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
except OSError:
    raise RuntimeError("Failed to launch the offline install for the codec wheels.") from None
if codec_result.returncode != 0:
    raise RuntimeError("Offline installation of the codec wheels failed.")

sys.path.insert(0, str(_src_root))

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

In [ ]:
import torch
from transformers import AutoModel

from knee_mri.dataset import split_labeled_studies, validated_plane_candidates
from knee_mri.image_model import (
    CONTINUOUS_DIMENSIONS,
    IMAGE_CLASSIFIER_C,
    PartialStandardScaler,
    bootstrap_macro_auc,
    build_image_classifier,
    cross_validate_image_model,
    fit_image_model,
    fold_signature,
    paired_bootstrap_delta,
    repeated_fold_macro_auc,
)
from knee_mri.intensity import load_processor_statistics
from knee_mri.labels import LABEL_COLUMNS
from knee_mri.laterality import DOMINANCE_GATE, SeriesLateralityEvidence, study_laterality
from knee_mri.metrics import macro_auc
from knee_mri.model_selection import select_multilabel_folds
from knee_mri.series_audit import audit_series
from knee_mri.slice_sampling import (
    MINIMUM_DECODED_SLICES,
    SLICE_SAMPLE_SIZE,
    select_plane_sample,
)
from knee_mri.study_features import (
    EMBEDDING_DIM,
    PLANES,
    STUDY_VECTOR_DIM,
    PlaneInput,
    build_study_features,
    max_pool,
    mean_max_pool,
    mean_pool,
    top_k_pool,
)
from knee_mri.submission import build_submission

## 1. Frozen Configuration

In [ ]:
IMAGE_MEAN, IMAGE_STD = load_processor_statistics(PROCESSOR_CONFIG_PATH)

frozen_contract = pd.Series(
    {
        "Study vector dimensions": STUDY_VECTOR_DIM,
        "Embedding dimensions": EMBEDDING_DIM,
        "Presence + reliability flags": STUDY_VECTOR_DIM - EMBEDDING_DIM,
        "Slices sampled per plane": SLICE_SAMPLE_SIZE,
        "Minimum decoded slices per plane": MINIMUM_DECODED_SLICES,
        "Laterality dominance gate": DOMINANCE_GATE,
        "Classifier C": IMAGE_CLASSIFIER_C,
        "Classifier penalty": build_image_classifier().estimator.penalty,
        "Classifier solver": build_image_classifier().estimator.solver,
        "Classifier class_weight": build_image_classifier().estimator.class_weight,
        "Classifier max_iter": build_image_classifier().estimator.max_iter,
        "Fold candidates": "(5, 4, 3, 2)",
        "Fold seed": SEED,
        "Scaled dimensions (flags unscaled)": CONTINUOUS_DIMENSIONS,
        "Processor image_mean": str(IMAGE_MEAN),
        "Processor image_std": str(IMAGE_STD),
    },
    name="Value",
).to_frame()

display(frozen_contract)

**Interpretation:** every value the pipeline depends on, read back from its frozen source rather than restated by hand, so a silent divergence between what this run did and what it was meant to do is visible here rather than inferred later from a score. `Laterality dominance gate` is a cost-asymmetry safety choice supported by measured coverage, not an empirical separation point — accepting a badly oblique acquisition would corrupt an input invisibly, while rejecting a usable one merely leaves it untransformed and flagged. `Classifier C` is deliberately `0.1` rather than the report model's `1.0`: that value suited 50,000 sparse text features, whereas this is roughly 388 dense features on the same 58 studies, a far higher per-feature overfitting risk. `Processor image_mean`/`image_std` are read from the attached model's own configuration; there is no remembered-constant fallback anywhere in this pipeline, because a plausible-but-wrong normalization is precisely the kind of error nothing downstream would catch.

## 2. Frozen Encoder and Environment

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Expected a GPU-enabled kernel for the image baseline.")


def _find_dinov2_dir(root: Path) -> Path:
    for config_path in root.rglob("config.json"):
        try:
            config = json.loads(config_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if config.get("model_type") == "dinov2":
            return config_path.parent
    raise RuntimeError("Could not find an attached DINOv2 model source.")


DEVICE = torch.device("cuda")
_dinov2_dir = _find_dinov2_dir(Path("/kaggle/input"))
dinov2 = AutoModel.from_pretrained(str(_dinov2_dir), local_files_only=True)
dinov2 = dinov2.to(DEVICE).eval()
for parameter in dinov2.parameters():
    parameter.requires_grad_(False)

cuda_major, cuda_minor = torch.cuda.get_device_capability(0)
GPU_COMPATIBLE = f"sm_{cuda_major}{cuda_minor}" in torch.cuda.get_arch_list()
if not GPU_COMPATIBLE:
    raise RuntimeError("Allocated GPU compute capability unsupported by installed PyTorch.")


# Returns the CLS token and the mean of the patch tokens side by side, from a
# single forward pass. The reported baseline uses the CLS half; the
# pre-registered patch-pooling variant uses the other. Computing both here
# rather than in two passes is what guarantees they differ only in the
# representation and in nothing upstream of it.
WIDE_EMBEDDING_DIM = 2 * EMBEDDING_DIM


def encode_batch(batch: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        outputs = dinov2(pixel_values=batch.to(DEVICE), interpolate_pos_encoding=True)
    hidden = outputs.last_hidden_state
    cls_token = hidden[:, 0, :]
    patch_mean = hidden[:, 1:, :].mean(dim=1)
    return torch.cat([cls_token, patch_mean], dim=1).detach().cpu()


# Smoke test: a checksum proves the bytes, not that the plugin loads.
codec_plugins = {
    name: importlib.util.find_spec(name) is not None
    for name in ("pylibjpeg", "libjpeg", "openjpeg")
}
if not all(codec_plugins.values()):
    raise RuntimeError("A vendored codec plugin failed to import after install.")

environment_summary = pd.Series(
    {
        "torch version": importlib.metadata.version("torch"),
        "transformers version": importlib.metadata.version("transformers"),
        "CUDA device": torch.cuda.get_device_name(0),
        "CUDA compute capability": f"{cuda_major}.{cuda_minor}",
        "DINOv2 parameters": sum(p.numel() for p in dinov2.parameters()),
        "Codec plugins importable": str(sorted(codec_plugins)),
        "Encoder trainable parameters": sum(
            p.numel() for p in dinov2.parameters() if p.requires_grad
        ),
    },
    name="Value",
).to_frame()

display(environment_summary)

**Interpretation:** records the exact runtime these scores were produced on, since a timing or numerical result is only meaningful alongside the hardware and library versions behind it. `Encoder trainable parameters` must be `0`: the encoder is frozen by design, and one that quietly trained would surface only as an unexplained score. An incompatible GPU raises an error rather than being skipped, because a partial run would still produce a submission — from an untested path.

## 3. Study Feature Extraction

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_series_df = pd.read_csv(DATA_DIR / "train_series.csv")
test_series_df = pd.read_csv(DATA_DIR / "test_series.csv")

labeled_studies, _ = split_labeled_studies(train_df)
labeled_studies = labeled_studies.reset_index(drop=True)

# Aggregate-only telemetry. Every entry is a count or a rate; no study or
# series identifier is ever placed in these structures.
telemetry = {
    "planes_absent": 0,
    "plane_retries": 0,
    "candidates_tried": 0,
    "decoded_slice_counts": [],
    "laterality_unreliable_studies": 0,
    "studies_with_no_plane": 0,
    "header_read_failures": 0,
}


def _study_laterality(series_df: pd.DataFrame, series_root: Path, study_id: str, sink=None):
    """Conservative consensus over EVERY available series in the study."""
    sink = telemetry if sink is None else sink
    evidence = []
    study_dir = series_root / study_id
    if not study_dir.is_dir():
        return study_laterality([])
    for series_dir in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        try:
            audit = audit_series(series_dir, decode_sample_size=1)
        except FileNotFoundError:
            continue
        sink["header_read_failures"] += audit.header_read_failures
        evidence.append(
            SeriesLateralityEvidence(
                tag=audit.laterality_tag,
                geometry=audit.laterality_from_geometry,
                cross_tag_conflict=audit.laterality_cross_tag_conflict,
            )
        )
    return study_laterality(evidence)


def build_features_for(
    series_df: pd.DataFrame,
    series_root: Path,
    study_ids,
    timings=None,
    per_plane=None,
    sample_size: int = SLICE_SAMPLE_SIZE,
    sink=None,
    slice_pool=mean_pool,
    pooled_width: int = EMBEDDING_DIM,
) -> np.ndarray:
    # A second extraction pass at a different density must not fold its
    # counters into the baseline's, or the reported telemetry becomes a
    # mixture of two contracts. The sink defaults to the baseline's.
    sink = telemetry if sink is None else sink
    vectors = []
    for study_id in study_ids:
        study_start = time.perf_counter()
        study_dir = series_root / study_id
        # Total .dcm files in the study is the real I/O surface: ordering
        # validation reads every header of every candidate series, not just
        # the five slices ultimately decoded.
        study_slices = len(list(study_dir.rglob("*.dcm"))) if study_dir.is_dir() else 0
        planes = {}
        for plane in PLANES:
            candidates = validated_plane_candidates(series_df, series_root, study_id, plane)
            sink["candidates_tried"] += len(candidates)
            outcome = select_plane_sample(
                [paths for _, paths in candidates], sample_size=sample_size
            )
            if outcome.absent or outcome.sample is None:
                sink["planes_absent"] += 1
                continue
            if outcome.candidates_tried > 1:
                sink["plane_retries"] += 1
            sink["decoded_slice_counts"].append(outcome.sample.decoded)
            winning_paths = candidates[outcome.candidates_tried - 1][1]
            header = pydicom.dcmread(winning_paths[0], stop_before_pixels=True)
            planes[plane] = PlaneInput(
                images=outcome.sample.images,
                image_orientation_patient=[float(v) for v in header.ImageOrientationPatient],
                pixel_spacing=[float(v) for v in header.PixelSpacing],
            )

        features = build_study_features(
            planes,
            _study_laterality(series_df, series_root, study_id, sink=sink),
            encode_batch,
            IMAGE_MEAN,
            IMAGE_STD,
            embedding_dim=WIDE_EMBEDDING_DIM,
            slice_pool=slice_pool,
        )
        if not features.laterality_reliable:
            sink["laterality_unreliable_studies"] += 1
        if not features.has_any_plane:
            sink["studies_with_no_plane"] += 1
        if timings is not None:
            timings.append(
                {
                    "slices": study_slices,
                    "seconds": time.perf_counter() - study_start,
                }
            )
        if per_plane is not None:
            per_plane.append(features.plane_embeddings)
        # Slice the reported baseline back out of the wide vector: the CLS
        # half plus the four flags, exactly the frozen 388-wide contract.
        vectors.append(
            np.concatenate(
                [
                    features.vector[:pooled_width],
                    features.vector[WIDE_EMBEDDING_DIM:],
                ]
            )
        )
    return np.vstack(vectors)


import pydicom  # noqa: E402  (imported after the source path is established)

labeled_timings = []
labeled_plane_embeddings = []
extraction_start = time.perf_counter()
train_features = pd.DataFrame(
    build_features_for(
        train_series_df,
        DATA_DIR / "train_series",
        labeled_studies["StudyInstanceUID"],
        timings=labeled_timings,
        per_plane=labeled_plane_embeddings,
    )
)
train_extraction_seconds = time.perf_counter() - extraction_start

In [ ]:
decoded_counts = pd.Series(telemetry["decoded_slice_counts"], dtype=float)

extraction_telemetry = pd.Series(
    {
        "Labeled studies": float(len(labeled_studies)),
        "Planes absent": float(telemetry["planes_absent"]),
        "Plane retries triggered": float(telemetry["plane_retries"]),
        "Candidates validated per plane (mean)": float(
            telemetry["candidates_tried"] / max(len(labeled_studies) * len(PLANES), 1)
        ),
        "Decoded slices per plane (mean)": (
            float(decoded_counts.mean()) if len(decoded_counts) else float("nan")
        ),
        "Decoded slices per plane (min)": (
            float(decoded_counts.min()) if len(decoded_counts) else float("nan")
        ),
        "Planes with fewer than five decoded": float((decoded_counts < SLICE_SAMPLE_SIZE).sum()),
        "Header read failures": float(telemetry["header_read_failures"]),
        "Studies with unreliable laterality": float(telemetry["laterality_unreliable_studies"]),
        "Studies with no usable plane": float(telemetry["studies_with_no_plane"]),
        "Feature extraction seconds": float(train_extraction_seconds),
    },
    name="Value",
).to_frame()

display(extraction_telemetry)

**Interpretation:** these counts are the only way several fallback paths become observable at all, because the sampled data has never exercised them. `Planes absent` and `Plane retries triggered` above zero would mean the same-plane retry and missing-plane fallback are doing real work rather than sitting unused. `Planes with fewer than five decoded` counts planes that fell back to a three- or four-slice mean. `Studies with unreliable laterality` is expected to be small but non-zero — roughly three of 58, predicted from the measured orientation distribution — and each such study is left untransformed with its flag set to 0 rather than flipped on a doubtful call. `Studies with no usable plane` must be **zero** for labeled training studies: a non-zero value is a data-path problem to diagnose, not a rounding detail.

## 4. Out-of-Fold Evaluation

In [ ]:
y = labeled_studies[LABEL_COLUMNS].astype(int).reset_index(drop=True)
selected_splits, folds = select_multilabel_folds(y, seed=SEED)

fold_identity = pd.Series(
    {
        "Labeled studies": len(labeled_studies),
        "Selected fold count": selected_splits,
        "Fold assignment signature": fold_signature(
            labeled_studies["StudyInstanceUID"].tolist(), folds
        ),
    },
    name="Value",
).to_frame()

display(fold_identity)

**Interpretation:** `select_multilabel_folds` is row-order-sensitive, so the same algorithm and seed do **not** by themselves imply the same fold membership as the report baseline. The signature digests the ordered study identifiers together with their fold assignment, so comparability between the two baselines is something this run demonstrates rather than assumes. The signature is a hash: it identifies an assignment without exposing which study went where.

In [ ]:
constant_predictions = pd.DataFrame(0.5, index=y.index, columns=LABEL_COLUMNS)
if macro_auc(y, constant_predictions) != 0.5:
    raise RuntimeError("Metric wiring check failed: a constant frame must score 0.5.")

cv_result = cross_validate_image_model(train_features, y, folds)

# The extraction path was restructured to emit two representations from one
# forward pass. The reported baseline must be unaffected by that: three
# earlier runs produced this score bit-identically, so a mismatch here means
# the restructuring changed the contract rather than merely widening it.
EXPECTED_BASELINE_MACRO_AUC = 0.6345688959
if abs(cv_result.pooled_macro_auc - EXPECTED_BASELINE_MACRO_AUC) > 1e-9:
    raise RuntimeError(
        "baseline macro AUC changed after the extraction was restructured"
    )

pooled_summary = pd.Series(
    {
        "Pooled OOF macro AUC": cv_result.pooled_macro_auc,
        "Fold macro AUC (mean)": float(np.mean(cv_result.fold_macro_auc)),
        "Fold macro AUC (min)": float(np.min(cv_result.fold_macro_auc)),
        "Fold macro AUC (max)": float(np.max(cv_result.fold_macro_auc)),
        "Constant-prediction sanity check": macro_auc(y, constant_predictions),
    },
    name="Value",
).to_frame()

display(pooled_summary)

**Interpretation:** `Pooled OOF macro AUC` is the primary metric and the only one that should be compared against the report baseline. The fold spread is **diagnostic only**: with 58 studies split across folds, individual fold scores are noisy, and a fold below 0.5 is not by itself evidence of a wiring error. The constant-prediction check is what would catch a genuine metric miswiring — it must be exactly 0.5, and the run aborts if it is not, because every score below it would otherwise be meaningless in a way no later inspection could detect.

In [ ]:
per_label_summary = (
    pd.Series(cv_result.pooled_per_label_auc, name="Pooled AUC")
    .reindex(LABEL_COLUMNS)
    .to_frame()
)
per_label_summary["Positives"] = [int(y[label].sum()) for label in LABEL_COLUMNS]

display(per_label_summary)

**Interpretation:** per-label AUC alongside the positive count that produced it, because a label with very few positives yields a score dominated by which side of a fold boundary those cases fell on. These are diagnostic: the pooled macro figure above is the contract's metric, and no per-label value here may be used to select a threshold, a feature, or a hyperparameter after the fact.

In [ ]:
# A single pooled score on 58 studies carries two distinct uncertainties, and
# neither is visible in the point estimate. The bootstrap holds the model
# fixed and asks how much the score moves under a different draw of studies.
# The repeated-fold pass asks the opposite: how much of the score is an
# artifact of which split the frozen seed produced.
#
# The frozen seed's score remains the reported result. The other seeds are a
# diagnostic and nothing may be selected from them -- picking the best would
# be precisely the optimistic bias this protocol exists to prevent.
DIAGNOSTIC_FOLD_SEEDS = (42, 43, 44, 45, 46, 47, 48, 49, 50, 51)
BOOTSTRAP_ITERATIONS = 2_000

interval = bootstrap_macro_auc(
    y, cv_result.oof_probabilities, iterations=BOOTSTRAP_ITERATIONS, seed=SEED
)
repeated_scores = repeated_fold_macro_auc(train_features, y, DIAGNOSTIC_FOLD_SEEDS)
repeated = pd.Series(repeated_scores, dtype=float)

uncertainty_summary = pd.Series(
    {
        "Pooled OOF macro AUC (frozen seed)": interval.point,
        "Bootstrap 95% lower": interval.lower,
        "Bootstrap 95% upper": interval.upper,
        "Bootstrap iterations": float(interval.iterations),
        "Resamples with all 12 labels estimable": interval.complete_label_fraction,
        "Repeated-fold seeds": float(len(DIAGNOSTIC_FOLD_SEEDS)),
        "Repeated-fold mean": float(repeated.mean()),
        "Repeated-fold std": float(repeated.std(ddof=1)),
        "Repeated-fold min": float(repeated.min()),
        "Repeated-fold max": float(repeated.max()),
    },
    name="Value",
).to_frame()

display(uncertainty_summary)

**Interpretation:** the point estimate alone cannot say whether this score is solid or fragile, and the two ways it could be fragile are different questions. The **bootstrap** holds the fitted model fixed and resamples studies, answering how much the score would move on another draw of 58 studies from the same population; a wide interval means the estimate is thin, not that the model is bad. `Resamples with all 12 labels estimable` below 1.0 is expected here — resampling with replacement can leave a rare label with no positives at all, and such a label is dropped from that resample rather than scored on one class, which widens the interval slightly and is reported rather than hidden. The **repeated-fold spread** asks the opposite question: how much of the reported number is an artifact of which particular split the frozen seed produced. A tight spread means the protocol is stable; a wide one means any single split's score, including the reported one, should be read with corresponding caution. The frozen seed's result stays the reported score — the other seeds exist to characterize it, and selecting the best of them would reintroduce exactly the optimistic bias this evaluation is built to avoid.

In [ ]:
flag_columns = train_features.iloc[:, CONTINUOUS_DIMENSIONS:]
flag_variance = pd.Series(
    {
        f"Flag {position} variance": float(flag_columns.iloc[:, position].var())
        for position in range(flag_columns.shape[1])
    },
    name="Value",
).to_frame()

display(flag_variance)

**Interpretation:** the four trailing flags are three plane-presence indicators and one laterality-reliability indicator. A variance of zero means the flag is constant across all 58 studies and therefore carries no information the head can learn from — its coefficient is unidentifiable from the intercept and is shrunk to approximately zero. This is the expected outcome given every audit measured complete plane coverage, and it is measured here rather than assumed. It does not indicate a fault: graceful degradation for a missing plane comes from excluding that plane from the mean, not from the flag.

## 4b. Pre-Registered Aggregation Variants

In [ ]:
# Round 46 deferred two aggregation alternatives as predefined experiments
# rather than silent fallbacks. Both are built from the SAME per-plane
# embeddings already extracted above, so no variant costs an extra decode or
# forward pass and none can differ by accident of extraction.
#
# The incumbent is favoured a priori: 58 studies cannot obviously support
# tripling the feature width or the head count. The registered decision rule
# is that V0 stands unless a paired delta excludes zero in a variant's favour.
def _concatenated_features(plane_embeddings, flags):
    """V1: keep plane identity instead of averaging it away."""
    rows = []
    for embeddings, flag_row in zip(plane_embeddings, flags, strict=True):
        # The encoder now returns CLS and patch-mean side by side. V1 and V2
        # were registered against the CLS half, so slice it explicitly --
        # taking the full width would silently redefine both variants and
        # make their numbers incomparable with the run that registered them.
        parts = [
            embeddings[plane][:EMBEDDING_DIM]
            if plane in embeddings
            else np.zeros(EMBEDDING_DIM, dtype=np.float64)
            for plane in PLANES
        ]
        rows.append(np.concatenate([*parts, flag_row]))
    return pd.DataFrame(np.vstack(rows))


flag_block = train_features.iloc[:, CONTINUOUS_DIMENSIONS:].to_numpy()
concat_features = _concatenated_features(labeled_plane_embeddings, flag_block)

# V1 uses the same head and folds, only a wider feature block.
concat_scaler_width = EMBEDDING_DIM * len(PLANES)
if concat_features.shape[1] != concat_scaler_width + flag_block.shape[1]:
    raise RuntimeError("V1 feature width does not match its registered definition")

# Through the shared harness rather than a hand-rolled fold loop. The
# hand-rolled version is where the round-97 defect lived: it carried its own
# scaler width, so widening the encoder left half the block unstandardized
# without failing. The harness validates fold coverage and derives the
# expected feature width from the scaler width it is given.
v1_result = cross_validate_image_model(
    concat_features, y, folds, continuous_dimensions=concat_scaler_width
)
v1_oof = v1_result.oof_probabilities

# V2: one head per plane, probabilities averaged over the planes present for
# each study. A study contributes to a plane's head only when it has that
# plane, so absent planes are excluded rather than zero-filled here too.
v2_accumulator = np.zeros((len(y), len(LABEL_COLUMNS)))
v2_counts = np.zeros((len(y), 1))
for plane in PLANES:
    present_mask = np.array(
        [plane in embeddings for embeddings in labeled_plane_embeddings]
    )
    plane_matrix = np.vstack(
        [
            embeddings[plane][:EMBEDDING_DIM]
            if plane in embeddings
            else np.zeros(EMBEDDING_DIM, dtype=np.float64)
            for embeddings in labeled_plane_embeddings
        ]
    )
    if plane_matrix.shape[1] != EMBEDDING_DIM:
        raise RuntimeError("V2 plane matrix width does not match its registered definition")
    for training_indices, validation_indices in folds:
        train_rows = training_indices[present_mask[training_indices]]
        validation_rows = validation_indices[present_mask[validation_indices]]
        if len(train_rows) == 0 or len(validation_rows) == 0:
            continue
        if y.iloc[train_rows].nunique().min() < 2:
            continue
        scaler = PartialStandardScaler(continuous_dimensions=EMBEDDING_DIM)
        scaler.fit(plane_matrix[train_rows])
        head = build_image_classifier()
        head.fit(scaler.transform(plane_matrix[train_rows]), y.iloc[train_rows])
        v2_accumulator[validation_rows] += head.predict_proba(
            scaler.transform(plane_matrix[validation_rows])
        )
        v2_counts[validation_rows] += 1

v2_oof = pd.DataFrame(
    np.divide(v2_accumulator, np.maximum(v2_counts, 1)),
    index=y.index,
    columns=LABEL_COLUMNS,
)

# V3: mean-pooled patch tokens instead of the CLS token. The per-label result
# showed the baseline carries effusion and cruciate findings but sits at
# chance for meniscus, patellofemoral and MCL -- small, localized structures.
# CLS is a single global summary; the patch mean retains more of the spatial
# evidence those findings depend on.
patch_features = pd.DataFrame(
    np.vstack(
        [
            np.concatenate(
                [
                    np.mean(
                        [emb[EMBEDDING_DIM:] for emb in embeddings.values()], axis=0
                    )
                    if embeddings
                    else np.zeros(EMBEDDING_DIM),
                    flag_row,
                ]
            )
            for embeddings, flag_row in zip(
                labeled_plane_embeddings, flag_block, strict=True
            )
        ]
    )
)
v3_result = cross_validate_image_model(patch_features, y, folds)

# Three comparisons now share one family. At an unadjusted 95% level the
# family-wise error would be 14.3%, and V3 was added after seeing V2 come
# close -- exactly the situation multiplicity control exists for. Bonferroni
# at alpha/3 widens every interval, including the two already reported, so no
# earlier conclusion is quietly improved by the correction.
FAMILY_SIZE = 3
ADJUSTED_PERCENTILES = (100 * (0.05 / FAMILY_SIZE) / 2, 100 * (1 - (0.05 / FAMILY_SIZE) / 2))
ADJUSTED_LEVEL = f"{100 * (1 - 0.05 / FAMILY_SIZE):.2f}%"
LOWER_COLUMN = f"Delta {ADJUSTED_LEVEL} lower"
UPPER_COLUMN = f"Delta {ADJUSTED_LEVEL} upper"

variant_delta_v1 = paired_bootstrap_delta(
    y, v1_oof, cv_result.oof_probabilities,
    iterations=BOOTSTRAP_ITERATIONS, seed=SEED, percentiles=ADJUSTED_PERCENTILES,
)
variant_delta_v2 = paired_bootstrap_delta(
    y, v2_oof, cv_result.oof_probabilities,
    iterations=BOOTSTRAP_ITERATIONS, seed=SEED, percentiles=ADJUSTED_PERCENTILES,
)
variant_delta_v3 = paired_bootstrap_delta(
    y, v3_result.oof_probabilities, cv_result.oof_probabilities,
    iterations=BOOTSTRAP_ITERATIONS, seed=SEED, percentiles=ADJUSTED_PERCENTILES,
)

variant_summary = pd.DataFrame(
    {
        "V0 shared mean (incumbent)": {
            "Macro AUC": cv_result.pooled_macro_auc,
            "Delta vs V0": 0.0,
            LOWER_COLUMN: float("nan"),
            UPPER_COLUMN: float("nan"),
            "Resolved": False,
        },
        "V1 plane concatenation": {
            "Macro AUC": v1_result.pooled_macro_auc,
            "Delta vs V0": variant_delta_v1.delta,
            LOWER_COLUMN: variant_delta_v1.lower,
            UPPER_COLUMN: variant_delta_v1.upper,
            "Resolved": variant_delta_v1.excludes_zero,
        },
        "V2 per-plane heads": {
            "Macro AUC": macro_auc(y, v2_oof),
            "Delta vs V0": variant_delta_v2.delta,
            LOWER_COLUMN: variant_delta_v2.lower,
            UPPER_COLUMN: variant_delta_v2.upper,
            "Resolved": variant_delta_v2.excludes_zero,
        },
        "V3 patch-token pooling": {
            "Macro AUC": v3_result.pooled_macro_auc,
            "Delta vs V0": variant_delta_v3.delta,
            LOWER_COLUMN: variant_delta_v3.lower,
            UPPER_COLUMN: variant_delta_v3.upper,
            "Resolved": variant_delta_v3.excludes_zero,
        },
    }
).T

display(variant_summary)

**Interpretation:** two aggregation alternatives, registered before this run and built from the same per-plane embeddings as the incumbent, so nothing here differs by accident of extraction. `Delta vs V0` is the paired difference in macro AUC — both variants score the same studies under the same folds, so per-study difficulty cancels and the difference is far better determined than either variant's own interval. `Resolved` is the only column that decides anything: it is true when the paired interval excludes zero, meaning the direction of the difference is established rather than merely observed. A variant with a higher `Macro AUC` but `Resolved` false has **not** beaten the incumbent — at 58 studies that is the expected outcome, and treating a point estimate as a win is precisely the selection bias this evaluation is built to avoid. Note also that a variant's own `Macro AUC` here is not an unbiased estimate of that variant, because it is being read off the same out-of-fold predictions used to compare it.

## 4c. Pre-Registered Sampling-Density Experiment

One comparison, registered before this run in a family of its own: does
sampling the same central band three times more densely change the score?

The baseline sees five slices of a median 166 — about one sample every 25
slices. A meniscal tear or a cruciate disruption spans a handful of contiguous
slices, so at that spacing the sampler can step over a finding entirely rather
than merely blur it. Density is the one dimension where more data per study
partially compensates for having only 58 of them.

In [ ]:
# E1: fifteen central-band slices instead of five. Registered in round 98
# before this run, as a single comparison in its own family -- sampling
# density, not aggregation -- so the interval is a plain paired 95%.
#
# Exactly one variable moves. The band is unchanged, so density rises without
# reaching toward the periphery; MINIMUM_DECODED_SLICES stays at 3, so
# plane-absence behaviour and therefore study membership are identical; and
# SLICE_SAMPLE_SIZE keeps its frozen value, so V0 above is untouched.
DENSE_SAMPLE_SIZE = 15

dense_telemetry = {
    "planes_absent": 0,
    "plane_retries": 0,
    "candidates_tried": 0,
    "decoded_slice_counts": [],
    "laterality_unreliable_studies": 0,
    "studies_with_no_plane": 0,
    "header_read_failures": 0,
}

dense_start = time.perf_counter()
dense_features = pd.DataFrame(
    build_features_for(
        train_series_df,
        DATA_DIR / "train_series",
        labeled_studies["StudyInstanceUID"],
        sample_size=DENSE_SAMPLE_SIZE,
        sink=dense_telemetry,
    )
)
dense_extraction_seconds = time.perf_counter() - dense_start

# Study membership must be identical, or the comparison is between different
# datasets rather than between two densities on one.
if dense_features.shape != train_features.shape:
    raise RuntimeError("E1 feature matrix shape differs from the baseline's")
if dense_telemetry["studies_with_no_plane"] != telemetry["studies_with_no_plane"]:
    raise RuntimeError("E1 changed which studies have a usable plane")
if dense_telemetry["planes_absent"] != telemetry["planes_absent"]:
    raise RuntimeError("E1 changed which planes are present")

dense_result = cross_validate_image_model(dense_features, y, folds)
dense_delta = paired_bootstrap_delta(
    y,
    dense_result.oof_probabilities,
    cv_result.oof_probabilities,
    iterations=BOOTSTRAP_ITERATIONS,
    seed=SEED,
)

# NOT a cost estimate, and deliberately not expressed as a ratio. The second
# pass reads files the first pass has already pulled into the page cache, so
# it is timed against a warm cache while the baseline was timed against a cold
# one -- which is how the first measurement of this came out at 0.51, i.e.
# three times the slices in half the time. The baseline's own extraction has
# also varied 119s to 179s across runs for identical work. Adoption cost has
# to come from a dedicated cold run at the new density; these two numbers are
# recorded only so that confound stays visible rather than being rediscovered.
density_seconds_uncomparable = {
    "baseline_cold_cache_seconds": train_extraction_seconds,
    "dense_warm_cache_seconds": dense_extraction_seconds,
    "comparable": False,
}

density_summary = pd.DataFrame(
    {
        "V0 five slices (incumbent)": {
            "Slices per plane": SLICE_SAMPLE_SIZE,
            "Decoded per plane (mean)": float(
                np.mean(telemetry["decoded_slice_counts"])
            ),
            "Macro AUC": cv_result.pooled_macro_auc,
            "Delta vs V0": 0.0,
            "Delta 95% lower": float("nan"),
            "Delta 95% upper": float("nan"),
            "Resolved": False,
            "Extraction seconds (cold cache)": train_extraction_seconds,
        },
        "E1 fifteen slices": {
            "Slices per plane": DENSE_SAMPLE_SIZE,
            "Decoded per plane (mean)": float(
                np.mean(dense_telemetry["decoded_slice_counts"])
            ),
            "Macro AUC": dense_result.pooled_macro_auc,
            "Delta vs V0": dense_delta.delta,
            "Delta 95% lower": dense_delta.lower,
            "Delta 95% upper": dense_delta.upper,
            "Resolved": dense_delta.excludes_zero,
            "Extraction seconds (warm cache, not comparable)": (
                dense_extraction_seconds
            ),
        },
    }
).T

display(density_summary)

dense_per_label = pd.DataFrame(
    {
        "V0 five slices": pd.Series(cv_result.pooled_per_label_auc),
        "E1 fifteen slices": pd.Series(dense_result.pooled_per_label_auc),
    }
).reindex(LABEL_COLUMNS)
dense_per_label["Change"] = (
    dense_per_label["E1 fifteen slices"] - dense_per_label["V0 five slices"]
)

display(dense_per_label)


**Interpretation:** the registered decision rule is that V0 stands unless E1's
interval excludes zero in its favour; a higher point estimate does not displace
it. The per-label breakdown is diagnostic only and was not registered as a
decision input — it is shown because the hypothesis concerned focal findings
specifically, so *where* any change lands is more informative than the macro.

The two extraction times are **not comparable to each other**: the second pass
reads files the first has already cached. They are reported so that confound
stays visible. Adoption cost needs a dedicated cold run at the new
density.

## 4d. Pre-Registered Selective Pooling

A study is a bag of slices, and a focal finding appears in only a few of them.
Mean pooling assumes the label is a property of the *average* slice; a
multiple-instance view assumes it is a property of the *most indicative* one.

For effusion — diffuse, present on most slices — the two assumptions nearly
agree. For a meniscal tear or a fracture they disagree completely, and those
are exactly the labels sitting near chance. Two selective operators are
compared against the uniform mean **at the same sampling density**, so the
operator is the only difference.

In [ ]:
# E2 and E3, registered in advance as a family of two. Both are compared
# against E1 -- the mean at the SAME density -- so the pooling operator is the
# only thing that differs. Density alone was already measured and is null.
#
# Both are parameter-free. Learned attention is deliberately excluded: it adds
# capacity to a labelled set that has failed to resolve 0.017 five times, and
# any gain could not be separated from overfitting.
POOL_FAMILY_SIZE = 2
POOL_PERCENTILES = (
    100 * (0.05 / POOL_FAMILY_SIZE) / 2,
    100 * (1 - (0.05 / POOL_FAMILY_SIZE) / 2),
)
POOL_LEVEL = f"{100 * (1 - 0.05 / POOL_FAMILY_SIZE):.2f}%"
TOP_K = 3

selective_pools = {
    "E2 max over slices": max_pool,
    f"E3 mean of top {TOP_K}": top_k_pool(TOP_K),
}

pool_rows = {
    "E1 mean (reference)": {
        "Macro AUC": dense_result.pooled_macro_auc,
        "Delta vs E1": 0.0,
        f"Delta {POOL_LEVEL} lower": float("nan"),
        f"Delta {POOL_LEVEL} upper": float("nan"),
        "Resolved": False,
    }
}
pool_per_label = {"E1 mean (reference)": pd.Series(dense_result.pooled_per_label_auc)}
pool_results = {"E1 mean (reference)": dense_result}

for name, pool in selective_pools.items():
    pool_sink = {key: (0 if not isinstance(v, list) else []) for key, v in telemetry.items()}
    pool_features = pd.DataFrame(
        build_features_for(
            train_series_df,
            DATA_DIR / "train_series",
            labeled_studies["StudyInstanceUID"],
            sample_size=DENSE_SAMPLE_SIZE,
            sink=pool_sink,
            slice_pool=pool,
        )
    )
    # Same studies, same planes, same slices -- only the operator differs.
    if pool_features.shape != dense_features.shape:
        raise RuntimeError(f"{name}: feature shape differs from the reference")
    if pool_sink["decoded_slice_counts"] != dense_telemetry["decoded_slice_counts"]:
        raise RuntimeError(f"{name}: decoded a different set of slices")
    # A pool that silently fell back to the mean would manufacture a null.
    if np.allclose(pool_features.to_numpy(), dense_features.to_numpy()):
        raise RuntimeError(f"{name}: produced features identical to the mean pool")

    pool_result = cross_validate_image_model(pool_features, y, folds)
    pool_delta = paired_bootstrap_delta(
        y,
        pool_result.oof_probabilities,
        dense_result.oof_probabilities,
        iterations=BOOTSTRAP_ITERATIONS,
        seed=SEED,
        percentiles=POOL_PERCENTILES,
    )
    pool_rows[name] = {
        "Macro AUC": pool_result.pooled_macro_auc,
        "Delta vs E1": pool_delta.delta,
        f"Delta {POOL_LEVEL} lower": pool_delta.lower,
        f"Delta {POOL_LEVEL} upper": pool_delta.upper,
        "Resolved": pool_delta.excludes_zero,
    }
    pool_per_label[name] = pd.Series(pool_result.pooled_per_label_auc)
    pool_results[name] = pool_result

pooling_summary = pd.DataFrame(pool_rows).T
display(pooling_summary)

pooling_per_label = pd.DataFrame(pool_per_label).reindex(LABEL_COLUMNS)
display(pooling_per_label)


**Interpretation:** the registered rule is that a selective pool must exclude
zero against the mean at the same density to count as an improvement; a higher
point estimate does not. Displacing the reported baseline is a separate
question and would need its own comparison.

The pair is what makes a null interpretable. A maximum over 384 dimensions is
upward-biased and outlier-dominated, so a null from it alone could mean either
that selectivity does not help or that the maximum is too noisy to tell. The
top-3 mean is selective but damped: **if it helps where the maximum does not,
the answer is noise; if neither helps, selectivity is wrong for this
representation** — which would be evidence about the frozen encoder rather than
about the bag-of-slices model.

## 4e. Pre-Registered Mean-and-Max Concatenation

The two operators suit different findings: the mean is the right estimator for
a finding present on most slices, the maximum for one present on few. Rather
than commit the whole twelve-label panel to one assumption, this gives the head
both and lets it weight them per label.

The cost is a doubled feature width on 58 studies. That is a real risk, not a
free option — a wider block already failed to justify itself once here.

In [ ]:
# E4: per plane, the slice mean and the slice maximum side by side. Two
# comparisons registered as one family -- against the mean, and against the
# maximum -- so Bonferroni at alpha/2.
#
# Comparing against BOTH is the point. Against the mean alone, E4 would almost
# certainly look good simply because it contains the maximum; the question
# that matters is whether keeping the mean as well buys anything over the
# maximum on its own, and only the second comparison asks that.
CONCAT_FAMILY_SIZE = 2
CONCAT_PERCENTILES = (
    100 * (0.05 / CONCAT_FAMILY_SIZE) / 2,
    100 * (1 - (0.05 / CONCAT_FAMILY_SIZE) / 2),
)
CONCAT_LEVEL = f"{100 * (1 - 0.05 / CONCAT_FAMILY_SIZE):.2f}%"

meanmax_sink = {key: (0 if not isinstance(v, list) else []) for key, v in telemetry.items()}
meanmax_features = pd.DataFrame(
    build_features_for(
        train_series_df,
        DATA_DIR / "train_series",
        labeled_studies["StudyInstanceUID"],
        sample_size=DENSE_SAMPLE_SIZE,
        sink=meanmax_sink,
        slice_pool=mean_max_pool(EMBEDDING_DIM),
        pooled_width=WIDE_EMBEDDING_DIM,
    )
)

# The width is the risk this variant carries, so it is asserted rather than
# assumed -- and the whole embedding block is what gets standardized. Leaving
# the scaler at its default here is precisely the round-97 defect.
MEANMAX_WIDTH = WIDE_EMBEDDING_DIM
if meanmax_features.shape[1] != MEANMAX_WIDTH + (STUDY_VECTOR_DIM - EMBEDDING_DIM):
    raise RuntimeError("E4 feature width does not match its registered definition")
if meanmax_sink["decoded_slice_counts"] != dense_telemetry["decoded_slice_counts"]:
    raise RuntimeError("E4 decoded a different set of slices")

# The two halves must be the mean and the maximum of the same slices, which is
# checkable directly against the features already built: the leading half must
# equal E1's embedding block and no half may equal the other.
meanmax_matrix = meanmax_features.to_numpy(dtype=np.float64)
if not np.allclose(meanmax_matrix[:, :EMBEDDING_DIM], dense_features.to_numpy()[:, :EMBEDDING_DIM]):
    raise RuntimeError("E4's leading half is not the mean pool it claims to be")
if np.allclose(meanmax_matrix[:, :EMBEDDING_DIM], meanmax_matrix[:, EMBEDDING_DIM:MEANMAX_WIDTH]):
    raise RuntimeError("E4's two halves are identical; the concatenation is degenerate")

meanmax_result = cross_validate_image_model(
    meanmax_features, y, folds, continuous_dimensions=MEANMAX_WIDTH
)

concat_rows = {}
for reference_name in ("E1 mean (reference)", "E2 max over slices"):
    delta = paired_bootstrap_delta(
        y,
        meanmax_result.oof_probabilities,
        pool_results[reference_name].oof_probabilities,
        iterations=BOOTSTRAP_ITERATIONS,
        seed=SEED,
        percentiles=CONCAT_PERCENTILES,
    )
    concat_rows[f"E4 vs {reference_name}"] = {
        "E4 macro AUC": meanmax_result.pooled_macro_auc,
        "Reference macro AUC": pool_results[reference_name].pooled_macro_auc,
        "Delta": delta.delta,
        f"Delta {CONCAT_LEVEL} lower": delta.lower,
        f"Delta {CONCAT_LEVEL} upper": delta.upper,
        "Resolved": delta.excludes_zero,
    }

concat_summary = pd.DataFrame(concat_rows).T
display(concat_summary)

concat_per_label = pd.DataFrame(
    {
        "E2 max": pd.Series(pool_results["E2 max over slices"].pooled_per_label_auc),
        "E4 mean and max": pd.Series(meanmax_result.pooled_per_label_auc),
    }
).reindex(LABEL_COLUMNS)
concat_per_label["Change"] = (
    concat_per_label["E4 mean and max"] - concat_per_label["E2 max"]
)

display(concat_per_label)


**Interpretation:** the comparison that carries the weight is the one against
the maximum, not the one against the mean. E4 contains the maximum, so beating
the mean would tell us little; the question is whether retaining the mean
alongside it earns its doubled width.

Width is the specific risk. Twice the features on 58 studies means twice the
coefficients under the same L2 penalty and the same fold structure, and a
wider block has already failed to justify itself once in this notebook. A
result near zero here is the expected outcome of that trade, not a surprise.

## 4f. Pre-Registered Displacement Comparison

Every measurement of max pooling so far has been against the mean *at its own
density*, never against the reported baseline. This asks the practical question
directly: does it displace the incumbent?

It is also the ninth comparison in this notebook's history, and max pooling is
being tested precisely **because** it won the earlier ones. Both facts are
handled explicitly below rather than left to the reader.

In [ ]:
# E5: max pooling at fifteen slices against the reported baseline. No new
# extraction -- both sets of out-of-fold probabilities already exist, so this
# is the same studies and the same folds throughout.
#
# Two levels are reported, and the difference between them is the point.
#
# The nominal level treats this as one comparison. It is not: it is the ninth
# in this notebook, and the candidate was chosen BECAUSE it won earlier ones.
# Testing the winner of a search at a nominal level is a winner's curse, and
# the delta below is optimistically biased as an estimate of what a variant
# chosen this way would score on new data.
#
# The registered rule therefore uses the conservative level. Displacing the
# figure this project reports is a claim that should survive the strict
# reading, not one that needs the generous reading to stand up.
PRIOR_COMPARISONS = 8
DISPLACEMENT_FAMILY = PRIOR_COMPARISONS + 1
STRICT_PERCENTILES = (
    100 * (0.05 / DISPLACEMENT_FAMILY) / 2,
    100 * (1 - (0.05 / DISPLACEMENT_FAMILY) / 2),
)
STRICT_LEVEL = f"{100 * (1 - 0.05 / DISPLACEMENT_FAMILY):.2f}%"

e2_probabilities = pool_results["E2 max over slices"].oof_probabilities

displacement_nominal = paired_bootstrap_delta(
    y, e2_probabilities, cv_result.oof_probabilities,
    iterations=BOOTSTRAP_ITERATIONS, seed=SEED,
)
displacement_strict = paired_bootstrap_delta(
    y, e2_probabilities, cv_result.oof_probabilities,
    iterations=BOOTSTRAP_ITERATIONS, seed=SEED, percentiles=STRICT_PERCENTILES,
)

displacement_summary = pd.DataFrame(
    {
        "Nominal 95% (one comparison)": {
            "Candidate macro AUC": pool_results["E2 max over slices"].pooled_macro_auc,
            "Baseline macro AUC": cv_result.pooled_macro_auc,
            "Delta": displacement_nominal.delta,
            "Lower": displacement_nominal.lower,
            "Upper": displacement_nominal.upper,
            "Excludes zero": displacement_nominal.excludes_zero,
            "Displaces baseline": False,
        },
        f"Registered {STRICT_LEVEL} (family of {DISPLACEMENT_FAMILY})": {
            "Candidate macro AUC": pool_results["E2 max over slices"].pooled_macro_auc,
            "Baseline macro AUC": cv_result.pooled_macro_auc,
            "Delta": displacement_strict.delta,
            "Lower": displacement_strict.lower,
            "Upper": displacement_strict.upper,
            "Excludes zero": displacement_strict.excludes_zero,
            "Displaces baseline": displacement_strict.excludes_zero,
        },
    }
).T

display(displacement_summary)


**Interpretation:** only the second row can displace the reported baseline, and
that was fixed before the run. The first row is shown so the gap between the
two readings is visible — if a result excludes zero nominally but not strictly,
that gap *is* the finding, and the honest conclusion is "suggestive, not
sufficient".

Two caveats survive even a strict resolution. The candidate combines a denser
sample with a different operator, so a positive result is the pair, not the
operator alone — though the density leg was separately measured and null. And
because this candidate was chosen for having won earlier comparisons, its own
macro AUC overstates what an equivalently-chosen variant would score on new
data. Neither caveat is dissolved by the interval; both would have to travel
with any number reported from here.

## 5. Refit and Test Inference

In [ ]:
scaler, classifier = fit_image_model(train_features, y)

inference_start = time.perf_counter()
test_features = pd.DataFrame(
    build_features_for(test_series_df, DATA_DIR / "test_series", test_df["StudyInstanceUID"])
)
test_probabilities = classifier.predict_proba(scaler.transform(test_features.to_numpy()))
inference_seconds = time.perf_counter() - inference_start

In [ ]:
# The 58 labeled studies are not a representative timing sample: they are a
# fixed cohort, and per-study cost scales with how many DICOM files the study
# holds. Draw a supplemental sample stratified by that count, spanning the
# whole train corpus, so the projection rests on the range the hidden set
# will actually contain rather than on one cohort's middle.
TIMING_SAMPLE_PER_STRATUM = 5
TIMING_STRATA = 5

study_slice_totals = (
    train_series_df.groupby("StudyInstanceUID")["SeriesInstanceUID"].count().rename("series")
)
labeled_ids = set(labeled_studies["StudyInstanceUID"])
candidate_pool = study_slice_totals.loc[~study_slice_totals.index.isin(labeled_ids)]
strata = pd.qcut(candidate_pool.rank(method="first"), TIMING_STRATA, labels=False)

timing_rng = np.random.default_rng(SEED)
stratified_ids = []
for stratum in range(TIMING_STRATA):
    members = candidate_pool.index[strata == stratum].to_numpy()
    take = min(TIMING_SAMPLE_PER_STRATUM, len(members))
    stratified_ids.extend(timing_rng.choice(members, size=take, replace=False).tolist())

stratified_timings = []
build_features_for(
    train_series_df,
    DATA_DIR / "train_series",
    stratified_ids,
    timings=stratified_timings,
)

all_timings = pd.DataFrame(labeled_timings + stratified_timings)
all_timings["stratum"] = pd.qcut(
    all_timings["slices"].rank(method="first"), TIMING_STRATA, labels=False
)

timing_by_stratum = all_timings.groupby("stratum").agg(
    studies=("seconds", "count"),
    median_slices=("slices", "median"),
    mean_seconds=("seconds", "mean"),
    max_seconds=("seconds", "max"),
)

display(timing_by_stratum)

**Interpretation:** per-study cost scales with how many DICOM files a study holds, because ordering validation reads every header of every candidate series — not just the five slices ultimately decoded. Timing only the labeled cohort would measure one narrow band of that range, so this pools it with a supplemental sample stratified across the whole training corpus. A flat profile across strata would mean cost is dominated by fixed per-study work; a rising one means slice count is the driver, and the largest stratum is the one that determines whether a runtime budget holds.

In [ ]:
# The safety margin is not decoration. Decode cost was separately measured as
# I/O-contention-sensitive, rising roughly 2.7x when other work shared the
# kernel, and this projection extrapolates linearly from a sample far smaller
# than the hidden set. 3x covers the measured contention effect with room to
# spare; it is applied to the mean, and the slowest observed stratum is shown
# alongside so a pessimistic reading is available without recomputing.
RUNTIME_SAFETY_MARGIN = 3.0
DOCUMENTED_HIDDEN_STUDIES = 1300
RUNTIME_BUDGET_HOURS = 9.0

mean_seconds_per_study = float(all_timings["seconds"].mean())
slowest_stratum_seconds = float(timing_by_stratum["mean_seconds"].max())

timing_summary = pd.Series(
    {
        "Studies timed": float(len(all_timings)),
        "Slices per study (median)": float(all_timings["slices"].median()),
        "Slices per study (max)": float(all_timings["slices"].max()),
        "Seconds per study (mean)": mean_seconds_per_study,
        "Seconds per study (slowest stratum)": slowest_stratum_seconds,
        "Projected hours, mean rate": mean_seconds_per_study * DOCUMENTED_HIDDEN_STUDIES / 3600,
        "Projected hours, slowest stratum": (
            slowest_stratum_seconds * DOCUMENTED_HIDDEN_STUDIES / 3600
        ),
        "Safety margin applied": RUNTIME_SAFETY_MARGIN,
        "Projected hours with margin": (
            mean_seconds_per_study * DOCUMENTED_HIDDEN_STUDIES / 3600 * RUNTIME_SAFETY_MARGIN
        ),
        "Runtime budget (hours)": RUNTIME_BUDGET_HOURS,
        "Test inference seconds (visible studies)": float(inference_seconds),
    },
    name="Value",
).to_frame()

display(timing_summary)

**Interpretation:** this measures the **complete** path — series selection, ordering validation, decode, normalization, framing, canonicalization, encoding and the head — which is what a runtime budget actually turns on. An encoder-only figure understates it substantially, because decode and selection dominate. Two projections are given deliberately: the mean rate, and the slowest stratum's rate as a pessimistic bound. The margin covers the separately-measured I/O contention effect, where decode nearly tripled when other work shared the kernel; a figure measured on an idle kernel would otherwise flatter a busy one. Even the margin-bearing projection should be read as an order-of-magnitude check, since it extrapolates linearly to a set far larger than anything timed here.

In [ ]:
submission = build_submission(sample_df, test_df["StudyInstanceUID"], test_probabilities)
submission.to_csv("/kaggle/working/submission.csv", index=False)

submission_summary = pd.Series(
    {
        "Submission rows": float(len(submission)),
        "Submission columns": float(submission.shape[1]),
        "Probability minimum": float(test_probabilities.min()),
        "Probability maximum": float(test_probabilities.max()),
    },
    name="Value",
).to_frame()

display(submission_summary)

**Interpretation:** the submission is written once, inside the kernel, in the competition's own row and column order. Row and column counts confirm its shape without exposing any prediction, and the probability range confirms the head produced calibrated-looking output rather than saturating at 0 or 1. Writing this file is **not** the same as submitting it — that remains a separate and deliberate step.

## 6. Persisted Aggregate Summary

In [ ]:
summary = {
    "frozen_contract": json.loads(frozen_contract.to_json()),
    "environment": json.loads(environment_summary.to_json()),
    "extraction_telemetry": json.loads(extraction_telemetry.to_json()),
    "fold_identity": json.loads(fold_identity.to_json()),
    "pooled_scores": json.loads(pooled_summary.to_json()),
    "uncertainty": json.loads(uncertainty_summary.to_json()),
    "aggregation_variants": json.loads(variant_summary.to_json()),
    "sampling_density": json.loads(density_summary.to_json()),
    "sampling_density_per_label": json.loads(dense_per_label.to_json()),
    "sampling_density_seconds": density_seconds_uncomparable,
    "selective_pooling": json.loads(pooling_summary.to_json()),
    "selective_pooling_per_label": json.loads(pooling_per_label.to_json()),
    "mean_max_concatenation": json.loads(concat_summary.to_json()),
    "mean_max_per_label": json.loads(concat_per_label.to_json()),
    "displacement": json.loads(displacement_summary.to_json()),
    "per_label_scores": json.loads(per_label_summary.to_json()),
    "flag_variance": json.loads(flag_variance.to_json()),
    "timing": json.loads(timing_summary.to_json()),
    "timing_by_stratum": json.loads(timing_by_stratum.to_json()),
    "submission_shape": json.loads(submission_summary.to_json()),
}

with open("/kaggle/working/image_baseline_summary.json", "w") as handle:
    json.dump(summary, handle, indent=2)

**Interpretation:** every aggregate above, gathered into one file so it can be retrieved from `/kaggle/working` after the run — Kaggle does not expose a notebook kernel's rendered output through its file API, only files written to the working directory and a plain log. Nothing in this file identifies a study or carries a per-study prediction.